In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predicting-employee-burnout-score-using-machine-learning/test_Kaggle_Comp.csv
/kaggle/input/predicting-employee-burnout-score-using-machine-learning/train_Kaggle_Comp.csv
/kaggle/input/predicting-employee-burnout-score-using-machine-learning/sample_submission_Kaggle_Comp.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
!pip install xgboost

In [4]:
from xgboost import XGBRegressor

In [5]:
df = pd.read_csv("/kaggle/input/predicting-employee-burnout-score-using-machine-learning/train_Kaggle_Comp.csv")
df

,employee_id,age,gender,education_level,experience_years,job_role,company_type,working_hours_per_day,projects_handled,overtime_hours,...,remote_work,team_size,manager_support,job_satisfaction,mental_fatigue_score,stress_level,sleep_hours,physical_activity,health_issues,burnout_score
0,10001,50,Male,Bachelor,0,Data Analyst,MNC,7,3,24,...,Yes,7,1,5,9.9,4,5.8,No,No,0.627
1,10002,36,Male,Bachelor,6,Data Analyst,Corporate,8,5,27,...,No,14,1,2,8.9,4,7.2,Yes,Yes,0.586
2,10003,29,Female,Bachelor,25,Project Manager,Startup,9,6,7,...,Yes,9,3,3,7.1,5,6.9,Yes,Yes,0.874
3,10004,42,Female,Master,6,HR Executive,Startup,8,5,19,...,No,15,4,5,7.9,3,4.8,Yes,Yes,0.507
4,10005,40,Male,Master,6,Data Analyst,Corporate,7,3,26,...,Yes,17,4,5,4.2,3,7.6,Yes,No,0.449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,13996,33,Male,PhD,20,Software Engineer,Startup,8,3,8,...,No,6,1,5,5.2,4,6.6,Yes,Yes,0.683
3996,13997,54,Male,Bachelor,18,ML Engineer,Corporate,10,1,14,...,No,17,5,5,9.9,4,6.2,Yes,No,0.550
3997,13998,49,Male,Bachelor,21,Data Analyst,Corporate,7,3,19,...,No,19,1,2,0.7,4,7.1,No,No,0.200
3998,13999,29,Female,Bachelor,19,Software Engineer,MNC,7,3,15,...,No,14,3,3,6.7,2,4.6,No,No,0.865


In [6]:
df.head()

,employee_id,age,gender,education_level,experience_years,job_role,company_type,working_hours_per_day,projects_handled,overtime_hours,...,remote_work,team_size,manager_support,job_satisfaction,mental_fatigue_score,stress_level,sleep_hours,physical_activity,health_issues,burnout_score
0,10001,50,Male,Bachelor,0,Data Analyst,MNC,7,3,24,...,Yes,7,1,5,9.9,4,5.8,No,No,0.627
1,10002,36,Male,Bachelor,6,Data Analyst,Corporate,8,5,27,...,No,14,1,2,8.9,4,7.2,Yes,Yes,0.586
2,10003,29,Female,Bachelor,25,Project Manager,Startup,9,6,7,...,Yes,9,3,3,7.1,5,6.9,Yes,Yes,0.874
3,10004,42,Female,Master,6,HR Executive,Startup,8,5,19,...,No,15,4,5,7.9,3,4.8,Yes,Yes,0.507
4,10005,40,Male,Master,6,Data Analyst,Corporate,7,3,26,...,Yes,17,4,5,4.2,3,7.6,Yes,No,0.449


In [7]:
df.shape

(4000, 23)

In [8]:
df.isnull().sum()

employee_id              0
age                      0
gender                   0
education_level          0
experience_years         0
job_role                 0
company_type             0
working_hours_per_day    0
projects_handled         0
overtime_hours           0
work_pressure_score      0
deadline_frequency       0
work_life_balance        0
remote_work              0
team_size                0
manager_support          0
job_satisfaction         0
mental_fatigue_score     0
stress_level             0
sleep_hours              0
physical_activity        0
health_issues            0
burnout_score            0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   employee_id            4000 non-null   int64  
 1   age                    4000 non-null   int64  
 2   gender                 4000 non-null   object 
 3   education_level        4000 non-null   object 
 4   experience_years       4000 non-null   int64  
 5   job_role               4000 non-null   object 
 6   company_type           4000 non-null   object 
 7   working_hours_per_day  4000 non-null   int64  
 8   projects_handled       4000 non-null   int64  
 9   overtime_hours         4000 non-null   int64  
 10  work_pressure_score    4000 non-null   int64  
 11  deadline_frequency     4000 non-null   object 
 12  work_life_balance      4000 non-null   int64  
 13  remote_work            4000 non-null   object 
 14  team_size              4000 non-null   int64  
 15  mana

In [10]:
employee_id = df["employee_id"]
y = df["burnout_score"]
X = df.drop(["employee_id","burnout_score"],axis = 1)

In [11]:
X.shape


(4000, 21)

In [12]:


cap_cols = [
    "overtime_hours",
    "working_hours_per_day",
    "projects_handled",
    "mental_fatigue_score",
    "stress_load"
]

for col in cap_cols:
    if col in X_train.columns:
        cap_value = X_train[col].quantile(0.99)
        X_train[col] = np.minimum(X_train[col], cap_value)
        X_test[col]  = np.minimum(X_test[col], cap_value)


NameError: name 'X_train' is not defined

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns


In [ ]:
print("Numerical Columns:")
print(numeric_features)

print("\nCategorical Columns:")
print(categorical_features)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])


In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
test_df = pd.read_csv("/kaggle/input/predicting-employee-burnout-score-using-machine-learning/test_Kaggle_Comp.csv")
test_df.head()

In [ ]:
test_employee_id = test_df["employee_id"]



In [ ]:
X_test = test_df.drop("employee_id", axis=1)


In [ ]:
from sklearn.pipeline import Pipeline
xgb_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=6,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=2.0,
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist"
)


In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", xgb_model)
])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)


predictions = 0.98 * predictions + 0.01


predictions = np.clip(predictions, 0, 1)


In [ ]:
submission = pd.DataFrame({
    "employee_id": test_employee_id.values,
    "burnout_score": test_predictions
})


In [ ]:
submission.to_csv("submission.csv", index=False)
print("submission.csv created")


In [ ]:
submission.head()